In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

import langchain
print(langchain.__version__)

import os

1.3.1


In [3]:
llm = OllamaLLM(model="llama3.2")

embeddings_model = OllamaEmbeddings(model="llama3.2")

In [4]:
# Carregar o pdf
pdf_link="os-sertoes.pdf"

loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()

In [6]:
# Separar em Chunks (Pedaços de documento)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pages)

In [ ]:
# Salvar no Vector DB - Chroma
db = Chroma.from_documents(
    chunks,
    embedding=embeddings_model,
    persist_directory="sertoes_index_naive_rag"
)

In [8]:
# Carregar DB
vectordb = Chroma(
    persist_directory="sertoes_index_naive_rag",
    embedding_function=embeddings_model
)

# Load Retriever
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 20,
        "lambda_mult": 0.7,
    }
)

In [ ]:
def ask(question):
    prompt = ChatPromptTemplate.from_template("""
    Responda a pergunta com base apenas no contexto abaixo:
    
    <context>
    {context}
    </context>
    
    Pergunta: {question}
    """)

    docs = retriever.invoke(question)
    for i, doc in enumerate(docs):
        print(f"--- Chunk {i+1} ---")
        print(doc.page_content)
        print(f"Metadata: {doc.metadata}")
        print()
    
    # Chain padrão LCEL
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain.invoke(question)

In [10]:
user_question = input("User: ")
answer = ask(user_question)
print("Answer: ", answer)

User:  Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?


--- Chunk 1 ---
os sertões | euclides da cunha
155
homens válidos, ágeis, inteligentes e bravos, vivendo de 
vaqueirice e pequena criação, vieram, pela lei fatal dos 
tempos, a fazer parte dos grandes fastos criminais do Ceará, 
em uma guerra de família. Seus êmulos foram os Araújos, 
que formavam uma família rica, filiada a outras das mais 
antigas do norte da província. Viviam na mesma região, 
tendo como sede principal a povoação de Boa Viagem, que 
demora cerca de dez léguas de Quixeramobim. Foi uma das 
lutas mais sangrentas dos sertões do Ceará, a que se travou 
entre estes dois grupos de homens, desiguais na fortuna e 
posição oficial, ambos embravecidos na prática das violên -
cias, e numerosos.
Assim começa o narrador consciencioso 49 breve notícia sobre a 
genealogia de Antônio Conselheiro.
Metadata: {'page_label': '155', 'trapped': '/False', 'start_index': 0, 'moddate': '2014-05-09T00:34:14-03:00', 'creationdate': '2014-05-09T00:33:41-03:00', 'page': 184, 'creator': 'Adobe I

--- Chunk 1 ---
os sertões | euclides da cunha
595
Baixas 
Ao fim de três horas de combate, tinham-se mobilizado dois 
mil homens sem efeito algum. As nossas baixas avultavam. 
Além de grande número de praças e oficiais de menor patente, 
baquearam mortos, logo pela manhã, o comandante do 29º, Major 
Queirós, e o da 5ª Brigada, Tenente-coronel Tupi Ferreira Caldas.
Tupi Caldas 
A deste originara raro lance de bravura. Os soldados do 30º 
idolatravam-no. Era uma rara vocação militar. Irrequieto, nervoso 
e impulsivo, o seu temperamento casava-se bem à vertigem das 
cargas e à rudeza das casernas. Nesta campanha mesmo jogara vá -
rias vezes a vida. Fora o comandante da vanguarda a 18 de julho; e 
depois daquele dia saíra indene dos mais mortíferos tiroteios. As 
balas tinham-no até então poupado, arranhando-o, rendando-lhe 
o chapéu, amolgando-lhe a chapa do talim. A última fulminou-o. 
Entrou por um dos braços, soerguido para sustentar o binóculo 
com que contemplava o assalto, e traspa